# 💧 AI-Based Rainwater Harvesting System and Storage Capacity Recommendation Using MLP Optimization

Predict useful annual rainwater supply for candidate tanks, optimize storage capacity, and recommend a system concept.

> Educational simulation only—not construction-ready drainage, plumbing, water-quality, or structural design.

👉 **Open the interactive companion:** [https://rainwater-harvesting-recommender.streamlit.app](https://rainwater-harvesting-recommender.streamlit.app/?stage=start)

## Complete workflow

Site inputs → daily water balance → synthetic examples → MLP surrogate → capacity optimization → system recommendation → sensitivity review.

## Interactive learning journey

- [The Rainwater Storage Decision](https://rainwater-harvesting-recommender.streamlit.app/?stage=problem) — Recommendation and Optimization
- [Six Design Inputs](https://rainwater-harvesting-recommender.streamlit.app/?stage=inputs) — Tabular Features
- [Daily Tank Water Balance](https://rainwater-harvesting-recommender.streamlit.app/?stage=balance) — Simulation Target
- [Simulated Harvesting Designs](https://rainwater-harvesting-recommender.streamlit.app/?stage=data) — Synthetic Training Dataset
- [Preparing Design Variables](https://rainwater-harvesting-recommender.streamlit.app/?stage=prepare) — Leakage-Safe Scaling
- [Estimating Useful Rainwater](https://rainwater-harvesting-recommender.streamlit.app/?stage=model) — MLP Regression
- [Checking the Utilization Model](https://rainwater-harvesting-recommender.streamlit.app/?stage=audit) — Regression Audit
- [Choosing Tank Capacity](https://rainwater-harvesting-recommender.streamlit.app/?stage=optimize) — Candidate Optimization
- [System Type and Design Review](https://rainwater-harvesting-recommender.streamlit.app/?stage=system) — Interpretable Recommendation Layer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,r2_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
FEATURES=["roof_area_m2","annual_rain_mm","daily_demand_L","runoff_coefficient","tank_cost_per_L","max_capacity_L","candidate_capacity_L"]

---
# 1. The Rainwater Storage Decision
### Phase 1 of 5 · The Storage Dilemma

## Part 1 · In civil engineering
A harvesting design must collect useful roof runoff without oversizing storage.

## Part 2 · The engineering challenge
A small tank overflows and misses demand; a large tank costs more and may add little benefit.

## Part 3 · Where the AI comes in
Predict utilization for candidate capacities, then optimize a declared engineering objective.

**Civil Engineering:** The Rainwater Storage Decision → **AI:** Recommendation and Optimization → `useful supply versus cost, shortage, and overflow`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=problem](https://rainwater-harvesting-recommender.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

The AI predicts **useful annual supply for each candidate capacity**. A separate optimizer then weighs shortage, overflow, cost, and space, keeping the engineering trade-off visible.

## Part 5 · What you just built

**In the notebook:** Separate water-yield prediction from tank and system selection.

**Takeaway:** Maximum tank size is not automatically the best design.

[Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Six Design Inputs](https://rainwater-harvesting-recommender.streamlit.app/?stage=inputs) ▶

---
# 2. Six Design Inputs
### Phase 1 of 5 · The Storage Dilemma

## Part 1 · In civil engineering
The site is represented by roof area, annual rainfall, daily non-potable demand, runoff coefficient, tank cost, and maximum installation capacity.

## Part 2 · The engineering challenge
Annual rainfall volume alone ignores timing, consumption, storage cycling, cost, and space.

## Part 3 · Where the AI comes in
Keep the six inputs interpretable and add candidate capacity when evaluating alternatives.

**Civil Engineering:** Six Design Inputs → **AI:** Tabular Features → `area, rainfall, demand, runoff, cost, available capacity`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=inputs](https://rainwater-harvesting-recommender.streamlit.app/?stage=inputs)

## Part 4 · The technical explanation

In [ ]:
site={"roof_area_m2":180,"annual_rain_mm":900,"daily_demand_L":650,"runoff_coefficient":.85,"tank_cost_per_L":6,"max_capacity_L":12000}
potential=site["roof_area_m2"]*site["annual_rain_mm"]*site["runoff_coefficient"]
print("Theoretical annual harvest:",f"{potential:,.0f} L");pd.Series(site)

## Part 5 · What you just built

**In the notebook:** Create and inspect the required site inputs.

**Takeaway:** Storage design depends on both annual volume and daily timing.

◀ [Previous: The Rainwater Storage Decision](https://rainwater-harvesting-recommender.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Daily Tank Water Balance](https://rainwater-harvesting-recommender.streamlit.app/?stage=balance) ▶

---
# 3. Daily Tank Water Balance
### Phase 2 of 5 · Building Water Evidence

## Part 1 · In civil engineering
Rain enters the tank on wet days, demand removes water daily, and excess above capacity becomes overflow.

## Part 2 · The engineering challenge
Theoretical annual collection does not reveal how much water can actually be used.

## Part 3 · Where the AI comes in
Simulate daily rainfall and tank operation to calculate annual supplied water, overflow, and shortage.

**Civil Engineering:** Daily Tank Water Balance → **AI:** Simulation Target → `inflow, demand, storage, overflow, shortage`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=balance](https://rainwater-harvesting-recommender.streamlit.app/?stage=balance)

## Part 4 · The technical explanation

In [ ]:
def simulate_balance(area,rain_mm,demand,runoff,capacity,seed=0):
 rng=np.random.default_rng(seed);wet=rng.random(365)<np.clip(.12+rain_mm/3500,.16,.65);weights=rng.gamma(1.4,1,365)*wet;daily_rain=rain_mm*weights/weights.sum();store=supply=overflow=shortage=0.
 for mm in daily_rain:
  inflow=area*mm*runoff;overflow+=max(store+inflow-capacity,0);store=min(capacity,store+inflow);used=min(store,demand);store-=used;supply+=used;shortage+=demand-used
 return supply,overflow,shortage
print(dict(zip(["supplied_L","overflow_L","shortage_L"],simulate_balance(180,900,650,.85,8000,1))))

## Part 5 · What you just built

**In the notebook:** Implement the physical water-balance target generator.

**Takeaway:** Useful supply is constrained by rainfall timing, storage, and demand.

◀ [Previous: Six Design Inputs](https://rainwater-harvesting-recommender.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Simulated Harvesting Designs](https://rainwater-harvesting-recommender.streamlit.app/?stage=data) ▶

---
# 4. Simulated Harvesting Designs
### Phase 2 of 5 · Building Water Evidence

## Part 1 · In civil engineering
Training cases span climate, roof, demand, runoff, cost, and tank-size combinations.

## Part 2 · The engineering challenge
A surrogate trained on narrow cases will extrapolate poorly to new sites.

## Part 3 · Where the AI comes in
Sample broad plausible ranges and reserve unseen site-capacity combinations for testing.

**Civil Engineering:** Simulated Harvesting Designs → **AI:** Synthetic Training Dataset → `thousands of sites and candidate capacities`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=data](https://rainwater-harvesting-recommender.streamlit.app/?stage=data)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);rows=[]
for i in range(5000):
 area=rng.uniform(40,600);rain=rng.uniform(300,2200);demand=rng.uniform(100,2500);runoff=rng.uniform(.5,.95);cost=rng.uniform(2,15);maximum=rng.uniform(3000,30000);cap=rng.uniform(1000,maximum);sup,over,short=simulate_balance(area,rain,demand,runoff,cap,i)
 rows.append([area,rain,demand,runoff,cost,maximum,cap,sup,over,short])
data=pd.DataFrame(rows,columns=FEATURES+["supplied_L","overflow_L","shortage_L"]);print(data.describe().T);data.head()

## Part 5 · What you just built

**In the notebook:** Generate simulated utilization examples and audit ranges.

**Takeaway:** Synthetic results teach method; local rainfall records are required for real design.

◀ [Previous: Daily Tank Water Balance](https://rainwater-harvesting-recommender.streamlit.app/?stage=balance) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Preparing Design Variables](https://rainwater-harvesting-recommender.streamlit.app/?stage=prepare) ▶

---
# 5. Preparing Design Variables
### Phase 3 of 5 · Learning Utilization

## Part 1 · In civil engineering
Input magnitudes range from coefficients below one to capacities of many thousands of litres.

## Part 2 · The engineering challenge
Raw scales destabilize training and full-dataset scaling leaks test information.

## Part 3 · Where the AI comes in
Split first and standardize with training-only statistics.

**Civil Engineering:** Preparing Design Variables → **AI:** Leakage-Safe Scaling → `split first, fit scaler on training data`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=prepare](https://rainwater-harvesting-recommender.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
train,temp=train_test_split(data,test_size=.30,random_state=SEED);val,test=train_test_split(temp,test_size=.50,random_state=SEED);scaler=StandardScaler().fit(train[FEATURES]);Xtr,Xva,Xte=scaler.transform(train[FEATURES]),scaler.transform(val[FEATURES]),scaler.transform(test[FEATURES]);ytr,yva,yte=train.supplied_L.to_numpy(),val.supplied_L.to_numpy(),test.supplied_L.to_numpy();print(Xtr.shape,Xva.shape,Xte.shape)

## Part 5 · What you just built

**In the notebook:** Prepare seven MLP inputs including candidate tank capacity.

**Takeaway:** The same scaler must be used for every optimized candidate.

◀ [Previous: Simulated Harvesting Designs](https://rainwater-harvesting-recommender.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Estimating Useful Rainwater](https://rainwater-harvesting-recommender.streamlit.app/?stage=model) ▶

---
# 6. Estimating Useful Rainwater
### Phase 3 of 5 · Learning Utilization

## Part 1 · In civil engineering
The optimizer needs a fast estimate for many possible tank capacities.

## Part 2 · The engineering challenge
Repeated full simulation can become expensive across many sites and scenarios.

## Part 3 · Where the AI comes in
Train an MLP surrogate to estimate annual useful water supplied.

**Civil Engineering:** Estimating Useful Rainwater → **AI:** MLP Regression → `7 -> Dense64 -> Dense32 -> annual supply`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=model](https://rainwater-harvesting-recommender.streamlit.app/?stage=model)

## Part 4 · The technical explanation

In [ ]:
net=Sequential([Input((7,)),Dense(64,activation="relu"),Dropout(.1),Dense(32,activation="relu"),Dense(1)]);net.compile(optimizer="adam",loss="mae");early=EarlyStopping(monitor="val_loss",patience=8,restore_best_weights=True);history=net.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=80,batch_size=64,callbacks=[early],verbose=0);pd.DataFrame(history.history).plot();plt.ylabel("MAE (L/year)");plt.grid(alpha=.2);plt.show();net.summary()

## Part 5 · What you just built

**In the notebook:** Train with early stopping and inspect learning curves.

**Takeaway:** The MLP approximates the simulator; it does not replace hydrologic evidence.

◀ [Previous: Preparing Design Variables](https://rainwater-harvesting-recommender.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Utilization Model](https://rainwater-harvesting-recommender.streamlit.app/?stage=audit) ▶

---
# 7. Checking the Utilization Model
### Phase 3 of 5 · Learning Utilization

## Part 1 · In civil engineering
Prediction errors affect demand coverage and the selected tank size.

## Part 2 · The engineering challenge
A good average score can hide bias for small tanks, dry climates, or high-demand sites.

## Part 3 · Where the AI comes in
Audit held-out errors and clip predictions to physical limits.

**Civil Engineering:** Checking the Utilization Model → **AI:** Regression Audit → `MAE, R², residuals, physical clipping`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=audit](https://rainwater-harvesting-recommender.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
pred=net.predict(Xte,verbose=0).ravel();physical_max=np.minimum(test.roof_area_m2*test.annual_rain_mm*test.runoff_coefficient,test.daily_demand_L*365);pred=np.clip(pred,0,physical_max);print("MAE:",mean_absolute_error(yte,pred));print("R²:",r2_score(yte,pred));plt.scatter(yte,pred,s=8,alpha=.3);plt.xlabel("Simulated supply");plt.ylabel("MLP supply");plt.grid(alpha=.2);plt.show()

## Part 5 · What you just built

**In the notebook:** Evaluate predictions and examine residuals by capacity.

**Takeaway:** Optimization is only as trustworthy as the surrogate in the candidate region.

◀ [Previous: Estimating Useful Rainwater](https://rainwater-harvesting-recommender.streamlit.app/?stage=model) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Choosing Tank Capacity](https://rainwater-harvesting-recommender.streamlit.app/?stage=optimize) ▶

---
# 8. Choosing Tank Capacity
### Phase 4 of 5 · Optimizing the Design

## Part 1 · In civil engineering
Feasible tank capacities are limited by installation space and budget.

## Part 2 · The engineering challenge
Weights and cost assumptions determine whether water savings justify extra storage.

## Part 3 · Where the AI comes in
Score every feasible candidate and show the full trade-off table rather than only the winner.

**Civil Engineering:** Choosing Tank Capacity → **AI:** Candidate Optimization → `shortage + overflow + cost + diminishing returns`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=optimize](https://rainwater-harvesting-recommender.streamlit.app/?stage=optimize)

## Part 4 · The technical explanation

In [ ]:
caps=np.arange(1000,site["max_capacity_L"]+1,1000);candidates=pd.DataFrame([{**site,"candidate_capacity_L":c} for c in caps]);supply=net.predict(scaler.transform(candidates[FEATURES]),verbose=0).ravel();potential=site["roof_area_m2"]*site["annual_rain_mm"]*site["runoff_coefficient"];annual_demand=site["daily_demand_L"]*365;supply=np.clip(supply,0,np.minimum(potential,annual_demand));candidates["supply_L"]=supply;candidates["demand_met"]=supply/annual_demand;candidates["overflow_L"]=np.maximum(potential-supply,0);candidates["shortage_L"]=np.maximum(annual_demand-supply,0);candidates["cost_INR"]=caps*site["tank_cost_per_L"];candidates["score"]=1-(.5*candidates.shortage_L/annual_demand+.2*candidates.overflow_L/potential+.3*candidates.cost_INR/candidates.cost_INR.max());best=candidates.loc[candidates.score.idxmax()];display(candidates[["candidate_capacity_L","demand_met","overflow_L","shortage_L","cost_INR","score"]].style.format({"demand_met":"{:.1%}","cost_INR":"₹{:,.0f}","score":"{:.3f}"}));print("Recommended tank:",f"{best.candidate_capacity_L:,.0f} L")

## Part 5 · What you just built

**In the notebook:** Optimize capacity and visualize the elbow in benefit versus cost.

**Takeaway:** The recommended capacity is conditional on declared weights and candidate steps.

◀ [Previous: Checking the Utilization Model](https://rainwater-harvesting-recommender.streamlit.app/?stage=audit) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: System Type and Design Review](https://rainwater-harvesting-recommender.streamlit.app/?stage=system) ▶

---
# 9. System Type and Design Review
### Phase 5 of 5 · Selecting and Reviewing

## Part 1 · In civil engineering
Tank capacity, surface space, rainfall, and overflow opportunity influence the system arrangement.

## Part 2 · The engineering challenge
System choice also requires ground conditions, water quality, mosquito control, first-flush, filtration, structural access, codes, and maintenance planning.

## Part 3 · Where the AI comes in
Apply transparent conditions, explain the reason, test sensitivity, and require professional approval.

**Civil Engineering:** System Type and Design Review → **AI:** Interpretable Recommendation Layer → `rooftop, modular, underground, or storage + recharge`

> 🎬 **See this illustrated and interactive:** [https://rainwater-harvesting-recommender.streamlit.app/?stage=system](https://rainwater-harvesting-recommender.streamlit.app/?stage=system)

## Part 4 · The technical explanation

In [ ]:
def choose_system(cap,max_cap,rain,overflow):
 if rain>=1100 and overflow>30000:return "Recharge + storage system","High rainfall and remaining overflow create a recharge opportunity."
 if cap<=3000:return "Small rooftop tank","The selected storage requirement is small."
 if cap>=8000 and max_cap<=12000:return "Underground storage tank","The selected capacity is large relative to available installation space."
 return "Modular storage tanks","Modular storage provides scalable above-ground capacity."
system,reason=choose_system(best.candidate_capacity_L,site["max_capacity_L"],site["annual_rain_mm"],best.overflow_L)
print("Recommended system:",system);print("Reason:",reason);print("Expected demand supplied:",f"{best.demand_met:.1%}")
print("Required review: local daily rainfall, first flush, filtration, water quality, mosquito control, overflow routing, soil/groundwater, structural loads, access, maintenance, codes, and professional approval.")

## Part 5 · What you just built

**In the notebook:** Produce the final system, capacity, metrics, reasons, and limitations.

**Takeaway:** The output is a preliminary concept recommendation, not construction-ready design.

◀ [Previous: Choosing Tank Capacity](https://rainwater-harvesting-recommender.streamlit.app/?stage=optimize) &nbsp;|&nbsp; [Project overview](https://rainwater-harvesting-recommender.streamlit.app/?stage=start)

---
# Final engineering conclusion

The MLP surrogate estimates useful annual supply across feasible tank capacities. The optimizer exposes the cost–shortage–overflow trade-off, and an interpretable layer recommends a system concept. Real design requires local time-series rainfall and full civil, water-quality, regulatory, and maintenance review.